# 06-02 디코더 전용 모델 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요. 학습 셀은 T4 GPU로 몇 분 걸려요.

In [ ]:
import json
import math
import os
import random
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 0. 준비: 06-01의 데이터 코드

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")
with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
sentences = sorted({s.strip() for d in data for s in (d["premise"], d["hypothesis"])})
random.seed(0)
random.shuffle(sentences)
text = "\n".join(sentences) + "\n"

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = chars
V = len(chars)
data_ids = torch.tensor([stoi[ch] for ch in text])
n = int(len(data_ids) * 0.9)
train_ids, val_ids = data_ids[:n], data_ids[n:]

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

def get_batch(ids, batch_size, block_size):
    starts = torch.randint(0, len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([ids[s : s + block_size] for s in starts])  # (B, block_size)
    y = torch.stack([ids[s + 1 : s + block_size + 1] for s in starts])  # (B, block_size)
    return x.to(device), y.to(device)

print("문자 수:", len(text), "어휘집 크기:", V)

## 1. 미래를 가리는 마스크

In [ ]:
T = 6
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))  # (T, T), 아래 삼각형만 True
print(causal.int())

torch.manual_seed(0)
scores = torch.randn(T, T)  # (T, T)
scores = scores.masked_fill(~causal, float("-inf"))
weights = F.softmax(scores, dim=-1)  # (T, T)
print(weights.round(decimals=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(causal, cmap="Greys_r")
axes[0].set_title("causal mask (white = can see)")
axes[1].imshow(weights, cmap="Oranges", vmin=0, vmax=1)
axes[1].set_title("attention weights with mask")
for ax in axes:
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
plt.show()

## 2. GPT 모델

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)  # q, k, v를 한 번에 계산해요
        self.out_proj = nn.Linear(d_model, d_model)
        causal = torch.tril(torch.ones(max_len, max_len, dtype=torch.bool))  # (max_len, max_len)
        self.register_buffer("causal", causal)  # 학습하지 않는 텐서. model.to(device)로 함께 옮겨져요

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)  # 각각 (B, T, C)
        q, k, v = (t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in (q, k, v))  # (B, H, T, d_head)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        scores = scores.masked_fill(~self.causal[:T, :T], float("-inf"))  # 미래는 가려요
        self.weights = F.softmax(scores, dim=-1)  # (B, H, T, T)
        out = (self.weights @ v).transpose(1, 2).reshape(B, T, C)  # (B, T, C)
        return self.out_proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.ff(self.norm2(x))  # (B, T, d_model)


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=192, n_heads=6, n_layers=4, max_len=64, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, max_len, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # 다음 토큰 점수

    def forward(self, ids):
        # ids: (B, T)
        T = ids.shape[1]
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(T, device=ids.device))  # (B, T, d_model)
        for block in self.blocks:
            x = block(x)  # (B, T, d_model)
        return self.lm_head(self.norm(x))  # (B, T, vocab_size)

torch.manual_seed(0)
model = GPT(V).to(device)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))
x, y = get_batch(train_ids, 4, model.max_len)
print(x.shape, "->", model(x).shape)

## 3. 미래를 정말 못 보는지 확인하기

In [ ]:
model.eval()
with torch.no_grad():
    a = torch.tensor([encode("오늘은 날씨가 좋다")], device=device)  # (1, 10)
    b = torch.tensor([encode("오늘은 날씨가 나쁘")], device=device)  # (1, 10), 뒤 두 글자만 달라요
    la, lb = model(a), model(b)
print("앞 8자리 출력이 같은가:", torch.allclose(la[:, :8], lb[:, :8], atol=1e-5))
print("마지막 자리 출력이 같은가:", torch.allclose(la[:, -1], lb[:, -1], atol=1e-5))
_ = model.train()  # 다시 학습 모드로

## 4. 학습

In [ ]:
@torch.no_grad()
def estimate_loss(model, ids, iters=50):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(ids, 64, model.max_len)
        losses.append(F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item())
    model.train()
    return sum(losses) / len(losses)

torch.manual_seed(0)
model = GPT(V).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
max_steps = 3000
history = []
start = time.time()
for step in range(max_steps + 1):
    if step % 250 == 0:
        val_loss = estimate_loss(model, val_ids)
        history.append((step, val_loss))
        print(f"step {step:4d}: val loss={val_loss:.3f} ({time.time() - start:.0f}s)")
    x, y = get_batch(train_ids, 64, model.max_len)
    logits = model(x)  # (64, 64, V)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), "gpt.pt")

In [ ]:
steps, vals = zip(*history)
plt.figure(figsize=(6, 3.3))
plt.plot(steps, vals, "o-", color="tab:red", label="GPT (val)")
plt.axhline(math.log(V), color="gray", ls=":", label="uniform")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## 5. 문장 생성하기

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=80, seed=0):
    model.eval()
    g = torch.Generator(device=device).manual_seed(seed)
    ids = torch.tensor([encode(prompt)], device=device)  # (1, T)
    for _ in range(max_new_tokens):
        context = ids[:, -model.max_len :]  # 최대 길이까지만 봐요
        logits = model(context)[:, -1, :]  # (1, V), 마지막 자리의 다음 토큰 점수
        probs = F.softmax(logits, dim=-1)  # (1, V)
        nxt = torch.multinomial(probs, 1, generator=g)  # (1, 1)
        ids = torch.cat([ids, nxt], dim=1)  # (1, T + 1)
        if itos[nxt.item()] == "\n":
            break
    return decode(ids[0].tolist())

for prompt in ["오늘", "정부는", "이 영화는"]:
    print(generate(model, prompt).strip())

In [ ]:
with torch.no_grad():
    model.eval()
    ids = torch.tensor([encode("학교에서 친구를 만났다")], device=device)
    model(ids)
    w = model.blocks[-1].attn.weights[0].mean(0).cpu()  # (T, T), 헤드 평균
plt.figure(figsize=(4.5, 4))
plt.imshow(w, cmap="Oranges")
plt.xlabel("key position")
plt.ylabel("query position")
plt.title("last block attention (mean of heads)")
plt.colorbar()
plt.show()